# Notebook 04: Create Embeddings

In this notebook, we will learn how to convert text into **embeddings** (numerical vectors).

## What You Will Learn

- What embeddings are
- Why we need embeddings in RAG
- How to use HuggingFace Embeddings
- How embeddings capture semantic meaning

## What Are Embeddings?

**Embeddings** are numerical representations of text. They convert words, sentences, or documents into arrays of numbers (vectors) that capture their **meaning**.

```
"The cat sat on the mat" -> [0.12, -0.45, 0.87, ...] (1536 numbers)
```

Key property: **Similar texts have similar embeddings.**

- "The cat sat on the mat" and "A feline rested on the rug" will have **similar** embeddings
- "The cat sat on the mat" and "Stock prices went up today" will have **different** embeddings

## Why Do We Need Embeddings in RAG?

When a user asks a question, we need to find the most relevant chunks from our document. We can't do keyword matching because:

- The user might use different words than the document
- We need **semantic search** (meaning-based, not exact match)

Embeddings allow us to:
1. Convert the user's question into a vector
2. Compare it with all chunk vectors
3. Find the most similar chunks

## HuggingFace Embeddings

We'll use `sentence-transformers/all-MiniLM-L6-v2` from HuggingFace. It's:
- Fast and lightweight
- Works well for short text chunks
- Free and open source

## Step 1: Import Required Libraries

In [1]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
import numpy as np

# HuggingFaceEmbeddings: wraps sentence-transformers models
# numpy: for working with vectors

C:\Users\Ahmed\AppData\Local\Temp\ipykernel_24076\861332890.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


## Step 2: Load and Split the PDF

We'll reuse the code from Notebooks 02 and 03.

In [2]:
# Load PDF
loader = PyPDFLoader("../data/sample.pdf")
pages = loader.load()

# Split into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)
chunks = text_splitter.split_documents(pages)

print(f"Total chunks: {len(chunks)}")

Total chunks: 20


## Step 3: Create the Embeddings Model

In [3]:
# Initialize HuggingFace Embeddings
# The first time this runs, it will download the model (~80 MB)
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},  # Use CPU (set to "cuda" for GPU)
    encode_kwargs={"normalize_embeddings": True}
)

print("Embeddings model loaded successfully!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\Ahmed\Downloads\ai_pdf_chat_assistant\ai_pdf_chat_assistant\venv\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Ahmed\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embeddings model loaded successfully!


## Step 4: Generate Embeddings for a Single Text

In [4]:
# Create embeddings for a single sentence
test_text = "The quick brown fox jumps over the lazy dog"

# Generate embedding vector
vector = embeddings.embed_query(test_text)

print(f"Embedding dimension: {len(vector)}")
print(f"First 10 values: {vector[:10]}")
print(f"Type: {type(vector)}")

Embedding dimension: 384
First 10 values: [0.035496827214956284, 0.06128626689314842, 0.05269210413098335, 0.07070505619049072, 0.03310137614607811, -0.0306695606559515, 0.006620652042329311, -0.06118330359458923, -0.0013259707484394312, 0.010645642876625061]
Type: <class 'list'>


## Step 5: Generate Embeddings for All Chunks

In [5]:
# Generate embeddings for all chunks
texts = [chunk.page_content for chunk in chunks]

# embed_documents creates embeddings for a list of texts
all_embeddings = embeddings.embed_documents(texts)

print(f"Total embeddings: {len(all_embeddings)}")
print(f"Dimension of each embedding: {len(all_embeddings[0])}")

# Store embeddings back in chunks
for i, chunk in enumerate(chunks):
    chunk.metadata["embedding_dimension"] = len(all_embeddings[i])

Total embeddings: 20
Dimension of each embedding: 384


## Step 6: Compare Embeddings (Semantic Similarity)

In [6]:
from numpy import dot
from numpy.linalg import norm

# Function to calculate cosine similarity between two vectors
def cosine_similarity(v1, v2):
    return dot(v1, v2) / (norm(v1) * norm(v2))

# Let's compare some embeddings
question = "What is machine learning?"
question_embedding = embeddings.embed_query(question)

# Compare with first few chunks
print(f"Question: '{question}'")
print(f"\nSimilarity scores:")
for i, chunk in enumerate(chunks[:10]):
    similarity = cosine_similarity(question_embedding, all_embeddings[i])
    print(f"  Chunk {i+1}: {similarity:.4f} - {chunk.page_content[:80]}...")

Question: 'What is machine learning?'

Similarity scores:
  Chunk 1: 0.5162 - Introduction to Artificial Intelligence
Chapter 1: What is Artificial Intelligen...
  Chunk 2: 0.3722 - gathered to explore the possibility of creating machines that could think. Since...
  Chunk 3: 0.6298 - performing any intellectual task that a human can do. Super AI refers to a hypot...
  Chunk 4: 0.5467 - statistical techniques to find patterns in data.
There are three main types of M...
  Chunk 5: 0.4581 - structures. Examples include customer segmentation and anomaly detection.
3. Rei...
  Chunk 6: 0.4968 - Introduction to Artificial Intelligence
Chapter 3: Deep Learning
Deep Learning i...
  Chunk 7: 0.3908 - Deep Learning has achieved remarkable success in several domains. In computer vi...
  Chunk 8: 0.5691 - Machine Learning requires manual feature engineering, where domain experts must ...
  Chunk 9: 0.3771 - the next layer. The network learns by adjusting the weights of these connections...
  Chun

## Step 7: Embed Multiple Documents at Once

In [7]:
# You can also embed multiple texts at once for efficiency
batch_texts = [
    "Artificial intelligence is transforming healthcare.",
    "Machine learning algorithms learn from data.",
    "The weather today is sunny and warm.",
    "Deep learning uses neural networks with many layers.",
    "I love eating pizza on weekends."
]

# Embed all at once
batch_embeddings = embeddings.embed_documents(batch_texts)

# Compare similarities
print("Semantic similarity comparison:")
for i, text1 in enumerate(batch_texts):
    for j, text2 in enumerate(batch_texts):
        if j > i:
            sim = cosine_similarity(batch_embeddings[i], batch_embeddings[j])
            print(f"  '{text1[:40]}...' vs '{text2[:40]}...': {sim:.4f}")

Semantic similarity comparison:
  'Artificial intelligence is transforming ...' vs 'Machine learning algorithms learn from d...': 0.3671
  'Artificial intelligence is transforming ...' vs 'The weather today is sunny and warm....': 0.0241
  'Artificial intelligence is transforming ...' vs 'Deep learning uses neural networks with ...': 0.3360
  'Artificial intelligence is transforming ...' vs 'I love eating pizza on weekends....': 0.0677
  'Machine learning algorithms learn from d...' vs 'The weather today is sunny and warm....': 0.0342
  'Machine learning algorithms learn from d...' vs 'Deep learning uses neural networks with ...': 0.4693
  'Machine learning algorithms learn from d...' vs 'I love eating pizza on weekends....': 0.0600
  'The weather today is sunny and warm....' vs 'Deep learning uses neural networks with ...': -0.0291
  'The weather today is sunny and warm....' vs 'I love eating pizza on weekends....': 0.1623
  'Deep learning uses neural networks with ...' vs 'I love eat

## Key Takeaways

1. **Embeddings** convert text into numerical vectors
2. **Similar texts** have similar (close) vectors
3. **Cosine similarity** measures how similar two vectors are
4. HuggingFace `all-MiniLM-L6-v2` creates 384-dimensional embeddings
5. We use embeddings to find relevant document chunks for a question

## The Math Behind It

- Each word/concept is mapped to a position in a high-dimensional space
- Related concepts are closer together
- Distance = 1 - cosine_similarity

## Next Steps

Proceed to **Notebook 05: ChromaDB** to learn how to store and search embeddings.